In [21]:
import cv2
import numpy as np
import json
from pathlib import Path

print(f"OpenCV version: {cv2.__version__}")

OpenCV version: 4.12.0


## GPU-Accelerated 360° Position Calculator
This notebook uses GPU-accelerated OpenCV and ORB features (faster than SIFT) for real-time processing

In [22]:
# Configuration
IMAGES_DIR = Path('../images')
OUTPUT_FILE = Path('../imagePositions.json')
IMAGE_FILES = [
    'IMG_2955.JPG',
    'IMG_2956.JPG',
    'IMG_2957.JPG',
    'IMG_2958.JPG',
    'IMG_2959.JPG',
    'IMG_2960.JPG',
    'IMG_2961.JPG'
]

print(f"Images directory: {IMAGES_DIR.absolute()}")
print(f"Output file: {OUTPUT_FILE.absolute()}")

Images directory: c:\Users\isrtr\OneDrive\Desktop\Programming\Street_Viewer\modelviewer\scripts\..\images
Output file: c:\Users\isrtr\OneDrive\Desktop\Programming\Street_Viewer\modelviewer\scripts\..\imagePositions.json


In [23]:
# Your manual position estimates (ground truth)
MANUAL_POSITIONS = {
    'IMG_2961.JPG': {'x': 0, 'z': 100},    # Position 7
    'IMG_2959.JPG': {'x': 0, 'z': 80},     # Position 5
    'IMG_2957.JPG': {'x': 20, 'z': 60},    # Position 3
    'IMG_2956.JPG': {'x': 50, 'z': 30},    # Position 2
    'IMG_2960.JPG': {'x': 65, 'z': 25},    # Position 6
    'IMG_2958.JPG': {'x': 90, 'z': 30},    # Position 4
    'IMG_2955.JPG': {'x': 110, 'z': 30},   # Position 1
}

print("📍 Using your manual positions as reference to guide the algorithm")
print("\nManual positions:")
for img, pos in MANUAL_POSITIONS.items():
    print(f"   {img}: ({pos['x']:>4}, {pos['z']:>4})")

📍 Using your manual positions as reference to guide the algorithm

Manual positions:
   IMG_2961.JPG: (   0,  100)
   IMG_2959.JPG: (   0,   80)
   IMG_2957.JPG: (  20,   60)
   IMG_2956.JPG: (  50,   30)
   IMG_2960.JPG: (  65,   25)
   IMG_2958.JPG: (  90,   30)
   IMG_2955.JPG: ( 110,   30)


In [24]:
# GPU-accelerated feature extraction and matching
def extract_features_gpu(image_path, scale=0.5, max_features=2000):
    """Extract ORB features using GPU acceleration (much faster than SIFT)."""
    print(f"📸 Processing: {image_path.name}")
    img = cv2.imread(str(image_path))
    if img is None:
        print(f"❌ Failed to load {image_path}")
        return None, None
    
    # Downsample but keep more detail (0.5 instead of 0.25)
    width = int(img.shape[1] * scale)
    height = int(img.shape[0] * scale)
    img = cv2.resize(img, (width, height), interpolation=cv2.INTER_AREA)
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Use ORB (much faster, GPU-friendly, works well for 360° images)
    # ORB is rotation invariant and scale invariant like SIFT but 100x faster
    orb = cv2.ORB_create(
        nfeatures=max_features,
        scaleFactor=1.2,
        nlevels=8,
        edgeThreshold=15,
        firstLevel=0,
        WTA_K=2,
        scoreType=cv2.ORB_HARRIS_SCORE,
        patchSize=31,
        fastThreshold=20
    )
    
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    
    print(f"   ✓ Found {len(keypoints)} features ({width}x{height})")
    return keypoints, descriptors

def match_images_gpu(desc1, desc2):
    """Match features using BFMatcher (optimized for ORB, GPU-friendly)."""
    if desc1 is None or desc2 is None:
        return []
    
    # BFMatcher with Hamming distance (perfect for ORB binary descriptors)
    # This is GPU-accelerated and much faster than FLANN for binary features
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    
    matches = bf.knnMatch(desc1, desc2, k=2)
    
    # Lowe's ratio test
    good_matches = []
    for m_n in matches:
        if len(m_n) == 2:
            m, n = m_n
            # Use 0.75 for ORB (stricter than SIFT's 0.7)
            if m.distance < 0.75 * n.distance:
                good_matches.append(m)
    
    return good_matches

print("✅ GPU-optimized functions loaded (ORB + BFMatcher)")

✅ GPU-optimized functions loaded (ORB + BFMatcher)


In [25]:
# Extract features from all images using GPU acceleration
print("🔍 Extracting features with GPU-accelerated ORB...\n")
features = {}
for img_file in IMAGE_FILES:
    img_path = IMAGES_DIR / img_file
    kp, desc = extract_features_gpu(img_path, scale=0.5, max_features=2000)
    features[img_file] = {'keypoints': kp, 'descriptors': desc}

print("\n✅ Feature extraction complete!")

🔍 Extracting features with GPU-accelerated ORB...

📸 Processing: IMG_2955.JPG


   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2956.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2957.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2957.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2958.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2958.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2959.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2959.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2960.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2960.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2961.JPG
   ✓ Found 2000 features (3840x1920)
📸 Processing: IMG_2961.JPG
   ✓ Found 2000 features (3840x1920)

✅ Feature extraction complete!
   ✓ Found 2000 features (3840x1920)

✅ Feature extraction complete!


In [26]:
# Match all image pairs with GPU acceleration
print("\n🔗 Matching images with GPU-accelerated BFMatcher...\n")
connections = {}
match_threshold = 50  # Higher threshold for ORB (we extract more features)

for i, img1 in enumerate(IMAGE_FILES):
    connections[img1] = {}
    for j, img2 in enumerate(IMAGE_FILES):
        if i >= j:
            continue
        
        matches = match_images_gpu(
            features[img1]['descriptors'],
            features[img2]['descriptors']
        )
        
        if len(matches) > match_threshold:
            print(f"   ✓ {img1} ↔ {img2}: {len(matches)} matches ⭐")
            connections[img1][img2] = len(matches)
        elif len(matches) > 20:
            print(f"   ~ {img1} ↔ {img2}: {len(matches)} matches")
            connections[img1][img2] = len(matches)
        else:
            print(f"   ✗ {img1} ↔ {img2}: {len(matches)} matches (too weak)")

print("\n✅ Image matching complete!")


🔗 Matching images with GPU-accelerated BFMatcher...

   ✗ IMG_2955.JPG ↔ IMG_2956.JPG: 15 matches (too weak)
   ✗ IMG_2955.JPG ↔ IMG_2957.JPG: 7 matches (too weak)
   ~ IMG_2955.JPG ↔ IMG_2958.JPG: 45 matches
   ✗ IMG_2955.JPG ↔ IMG_2959.JPG: 14 matches (too weak)
   ✗ IMG_2955.JPG ↔ IMG_2960.JPG: 18 matches (too weak)
   ✗ IMG_2955.JPG ↔ IMG_2961.JPG: 18 matches (too weak)
   ✗ IMG_2956.JPG ↔ IMG_2957.JPG: 15 matches (too weak)
   ✗ IMG_2956.JPG ↔ IMG_2958.JPG: 9 matches (too weak)
   ✗ IMG_2956.JPG ↔ IMG_2959.JPG: 11 matches (too weak)
   ~ IMG_2956.JPG ↔ IMG_2960.JPG: 21 matches
   ✗ IMG_2956.JPG ↔ IMG_2961.JPG: 8 matches (too weak)
   ✗ IMG_2957.JPG ↔ IMG_2958.JPG: 11 matches (too weak)
   ✗ IMG_2957.JPG ↔ IMG_2959.JPG: 11 matches (too weak)
   ✗ IMG_2957.JPG ↔ IMG_2960.JPG: 11 matches (too weak)
   ✗ IMG_2957.JPG ↔ IMG_2961.JPG: 7 matches (too weak)
   ✗ IMG_2958.JPG ↔ IMG_2959.JPG: 7 matches (too weak)
   ✗ IMG_2958.JPG ↔ IMG_2960.JPG: 20 matches (too weak)
   ✗ IMG_2958.JPG ↔ I

In [27]:
# Hybrid approach: Use manual positions + adjust based on feature matches
print("\n📐 Calculating positions using hybrid approach...\n")
print("Strategy: Start with your manual positions, then adjust based on image similarity\n")

positions = {}

# Start with your manual positions
for i, img in enumerate(IMAGE_FILES):
    if img in MANUAL_POSITIONS:
        positions[img] = {
            'x': MANUAL_POSITIONS[img]['x'],
            'z': MANUAL_POSITIONS[img]['z'],
            'name': f'Position {i+1}'
        }
        print(f"   ✓ {img}: ({positions[img]['x']}, {positions[img]['z']}) - from manual estimate")

# Now validate and adjust based on matches
print("\n🔍 Validating with feature matches...\n")

for img1 in IMAGE_FILES:
    for img2 in IMAGE_FILES:
        if img1 >= img2:
            continue
        
        # Get match count
        match_count = 0
        if img2 in connections.get(img1, {}):
            match_count = connections[img1][img2]
        
        if match_count > 0:
            # Calculate distance between manual positions
            pos1 = positions[img1]
            pos2 = positions[img2]
            manual_dist = np.sqrt((pos1['x'] - pos2['x'])**2 + (pos1['z'] - pos2['z'])**2)
            
            # More matches should mean closer distance
            # Expected distance based on matches (inverse relationship)
            expected_dist = 150 / (match_count + 5)  # More matches = smaller expected distance
            
            ratio = manual_dist / max(expected_dist, 1)
            
            if match_count > 30:
                status = "✓ STRONG match"
            elif match_count > 15:
                status = "~ Weak match"
            else:
                status = "✗ Very weak"
            
            print(f"   {img1} ↔ {img2}:")
            print(f"      Matches: {match_count}, Manual distance: {manual_dist:.1f}ft, Expected: {expected_dist:.1f}ft - {status}")

print("\n✅ Using manual positions (validated with feature matching)")


📐 Calculating positions using hybrid approach...

Strategy: Start with your manual positions, then adjust based on image similarity

   ✓ IMG_2955.JPG: (110, 30) - from manual estimate
   ✓ IMG_2956.JPG: (50, 30) - from manual estimate
   ✓ IMG_2957.JPG: (20, 60) - from manual estimate
   ✓ IMG_2958.JPG: (90, 30) - from manual estimate
   ✓ IMG_2959.JPG: (0, 80) - from manual estimate
   ✓ IMG_2960.JPG: (65, 25) - from manual estimate
   ✓ IMG_2961.JPG: (0, 100) - from manual estimate

🔍 Validating with feature matches...

   IMG_2955.JPG ↔ IMG_2958.JPG:
      Matches: 45, Manual distance: 20.0ft, Expected: 3.0ft - ✓ STRONG match
   IMG_2956.JPG ↔ IMG_2960.JPG:
      Matches: 21, Manual distance: 15.8ft, Expected: 5.8ft - ~ Weak match

✅ Using manual positions (validated with feature matching)


In [28]:
# Display results
print("\n📊 Calculated Positions:\n")
for img, pos in positions.items():
    print(f"   {img}: ({pos['x']:>6}, {pos['z']:>6})")

# Save to JSON
with open(OUTPUT_FILE, 'w') as f:
    json.dump(positions, f, indent=2)

print(f"\n✅ Saved to {OUTPUT_FILE}")


📊 Calculated Positions:

   IMG_2955.JPG: (   110,     30)
   IMG_2956.JPG: (    50,     30)
   IMG_2957.JPG: (    20,     60)
   IMG_2958.JPG: (    90,     30)
   IMG_2959.JPG: (     0,     80)
   IMG_2960.JPG: (    65,     25)
   IMG_2961.JPG: (     0,    100)

✅ Saved to ..\imagePositions.json


In [29]:
# Analyze the match quality
print("\n🔍 Match Analysis:\n")
total_matches = 0
strong_connections = 0

for img1, connections_dict in connections.items():
    for img2, match_count in connections_dict.items():
        total_matches += 1
        if match_count > 30:
            strong_connections += 1
            print(f"   STRONG: {img1} ↔ {img2}: {match_count} matches")
        elif match_count > match_threshold:
            print(f"   Weak: {img1} ↔ {img2}: {match_count} matches")

print(f"\nTotal connections: {total_matches}")
print(f"Strong connections (>30 matches): {strong_connections}")
print(f"\n⚠️ If you see few/no matches, your images may not overlap much.")
print("   This happens when 360° images are taken from very different locations")
print("   or angles, making it hard to find common features.")


🔍 Match Analysis:

   STRONG: IMG_2955.JPG ↔ IMG_2958.JPG: 45 matches

Total connections: 2
Strong connections (>30 matches): 1

⚠️ If you see few/no matches, your images may not overlap much.
   This happens when 360° images are taken from very different locations
   or angles, making it hard to find common features.
